# Rectangular Wall Loss Strategy (interactive quick start)

This notebook shows how to use `particula.dynamics.RectangularWallLossStrategy`
to compute wall loss rates and apply time stepping in a box-shaped chamber.
The full published duration is retained with explicit positivity substeps.

## Prerequisites
- Python 3.12+
- `particula` installed (pip or conda)
- NumPy for light array handling

To run locally:
```bash
Use the selected E9 checkout: `python -m pip install -e ".[extra]"`
```

## Imports and helpers

In [1]:
import numpy as np
import particula as par

np.set_printoptions(precision=3, suppress=True)

## 1) Build a discrete particle distribution
We use the preset radius-binned distribution to keep setup minimal.

In [2]:
particle = par.particles.create_lognormal_binned_particle_data(
    distribution_type="discrete"
)
print("Bins:", particle.radii[0].size)
print(
    "Initial total concentration:",
    f"{particle.number_density[0]:.3e} 1/m^3",
)

Bins: 250
Initial total concentration: 1.100e+10 1/m^3


## 2) Configure `RectangularWallLossStrategy`
Dimensions are given as `(x, y, z)` in meters.

In [3]:
wall_loss = par.dynamics.RectangularWallLossStrategy(
    wall_eddy_diffusivity=1e-3,
    chamber_dimensions=(1.0, 1.2, 0.8),
    distribution_type="discrete",
)
T = 298.15  # K
P = 101325.0  # Pa

## 3) Compute instantaneous wall loss rate
This returns a per-bin concentration loss rate (m^-3 s^-1).

In [4]:
rate = wall_loss.rate(particle=particle, temperature=T, pressure=P)
print("Rate shape:", rate.shape)
print("First three rates (m^-3 s^-1):", rate.ravel()[:3])

Rate shape: (250,)
First three rates (m^-3 s^-1): [-0. -0. -0.]


## 4) Apply a short transient (30 min, 5 s steps)
Keeps runtime small while showing concentration decay.

In [5]:
dt = 5.0  # seconds
steps = int((30 * 60) / dt)
# Native Euler updates reject negative concentrations. Retain the output
# interval and full duration, but bound each internal step by max(k) * dt < 1.
sub_steps = max(
    1,
    int(
        np.ceil(
            np.max(
                wall_loss.loss_coefficient(particle, temperature=T, pressure=P)
            )
            * dt
            / 0.9
        )
    ),
)
print("Internal substeps per output interval:", sub_steps)
initial_counts = particle.concentration.copy()
initial_masses = particle.masses.copy()
loss_coefficients = wall_loss.loss_coefficient(
    particle, temperature=T, pressure=P
)
history = np.zeros(steps + 1)
history[0] = particle.number_density[0]

for i in range(1, steps + 1):
    for _substep in range(sub_steps):
        particle = wall_loss.step(
            particle=particle,
            temperature=T,
            pressure=P,
            time_step=dt / sub_steps,
        )
    history[i] = particle.number_density[0]

# Fixed radii give an independent repeated-Euler decay prediction.
np.testing.assert_allclose(
    particle.concentration,
    initial_counts
    * (1 - loss_coefficients * dt / sub_steps) ** (steps * sub_steps),
    rtol=1e-12,
    atol=1e-30,
)
np.testing.assert_array_equal(particle.masses, initial_masses)
assert np.all(particle.concentration >= 0)

print("Final concentration:", f"{history[-1]:.3e} 1/m^3")
print("Relative decay:", f"{history[-1] / history[0]:.3f}")

Internal substeps per output interval: 9
Final concentration: 1.065e+10 1/m^3
Relative decay: 0.968


### Optional: quick visualization
Uncomment to view a normalized decay curve (kept lightweight).

In [6]:
# from matplotlib import pyplot as plt
# minutes = np.arange(steps + 1) * dt / 60.0
# plt.plot(minutes, history / history[0])
# plt.xlabel("Time (min)")
# plt.ylabel("Normalized concentration")
# plt.title("Rectangular wall loss (1.0 x 1.2 x 0.8 m)")
# plt.grid(True)
# plt.tight_layout()
# plt.show()

## 5) Notes on other distribution types
The same API works for `"continuous_pdf"` and `"particle_resolved"`.
Only the underlying particle representation changes.

In [7]:
# Continuous PDF example (same call pattern)
wall_loss_pdf = par.dynamics.RectangularWallLossStrategy(
    wall_eddy_diffusivity=1e-3,
    chamber_dimensions=(1.0, 1.2, 0.8),
    distribution_type="continuous_pdf",
)
particle_pdf = par.particles.create_lognormal_binned_particle_data(
    distribution_type="continuous_pdf"
)
rate_pdf = wall_loss_pdf.rate(particle_pdf, temperature=T, pressure=P)
assert np.all(np.isfinite(rate_pdf)) and np.all(rate_pdf <= 0)

# Particle-resolved example
np.random.seed(42)
resolved = par.particles.create_lognormal_resolved_particle_data()
wall_loss_resolved = par.dynamics.RectangularWallLossStrategy(
    wall_eddy_diffusivity=1e-3,
    chamber_dimensions=(1.0, 1.2, 0.8),
    distribution_type="particle_resolved",
)
# resolved = wall_loss_resolved.step(
#     particle=resolved, temperature=T, pressure=P, time_step=dt
# )

## Summary
- Configure rectangular chambers with `chamber_dimensions` (x, y, z).
- Use `rate` for concentration loss rates; `loss_coefficient` gives 1/s.
- Use `step` to update particle concentrations over time.
- Switch `distribution_type` to target discrete, continuous PDF, or
  particle-resolved representations.